In [1]:
import pandas as pd

## Read Files

In [2]:
import pandas as pd
import os
import re

# Root folder
root_folder = r"C:\Users\Shubham Verma\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\Pattex\Ads\pattex\vendor_central\input"

spap_dfs = []
sb_dfs = []

# Date pattern from filename
date_pattern = re.compile(r'(\d{2}-\d{2}-\d{4})')

for root, dirs, files in os.walk(root_folder):

    for file in files:

        if not file.lower().endswith(".xlsx"):
            continue

        file_path = os.path.join(root, file)

        # Extract date from filename
        date_match = date_pattern.search(file)

        if not date_match:
            print(f"Date not found in filename: {file}")
            continue

        file_date = pd.to_datetime(
            date_match.group(1),
            format="%d-%m-%Y"
        )

        try:
            xls = pd.ExcelFile(file_path)

            # ==========================
            # Sponsored Products Campaigns
            # ==========================
            if "Sponsored Products Campaigns" in xls.sheet_names:

                df = pd.read_excel(
                    file_path,
                    sheet_name="Sponsored Products Campaigns"
                )

                if "Entity" in df.columns:

                    mask = (
                        df["Entity"]
                        .astype(str)
                        .str.strip()
                        .str.lower()
                        .eq("product ad")
                    )

                    df = df[mask].copy()
                    df["Date"] = file_date

                    spap_dfs.append(df)

            # ==========================
            # Sponsored Brands Campaigns
            # ==========================
            if "Sponsored Brands Campaigns" in xls.sheet_names:

                df = pd.read_excel(
                    file_path,
                    sheet_name="Sponsored Brands Campaigns"
                )

                if "Entity" in df.columns:

                    mask = (
                        df["Entity"]
                        .astype(str)
                        .str.strip()
                        .str.lower()
                        .eq("campaign")
                    )

                    df = df[mask].copy()
                    df["Date"] = file_date

                    sb_dfs.append(df)

        except Exception as e:
            print(f"Error in {file_path}")
            print(e)

# Final DataFrames
df_spap = (
    pd.concat(spap_dfs, ignore_index=True)
    if spap_dfs
    else pd.DataFrame()
)

df_sb = (
    pd.concat(sb_dfs, ignore_index=True)
    if sb_dfs
    else pd.DataFrame()
)

print("df_spap shape:", df_spap.shape)
print("df_sb shape:", df_sb.shape)

c:\Users\Shubham Verma\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Shubham Verma\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Shubham Verma\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Shubham Verma\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Shubham Verma\anaconda3\Lib\site-packages\openpyxl\styles\styleshee

df_spap shape: (2186, 53)
df_sb shape: (80, 52)


In [3]:
cols_to_use = [
    "Date",
    "ASIN",
    "Campaign Name",
    "Impressions",
    "Clicks",
    "Conversion Rate",
    "Spend",
    "Sales",
    "Orders",
    "Units",
    "CPC",
    "ROAS",
    "ACOS", 
]
df_spap = df_spap[cols_to_use].copy()
df_spap["Report_Type"] = "Sponsored Products"
df_spap

,Date,ASIN,Campaign Name,Impressions,Clicks,Conversion Rate,Spend,Sales,Orders,Units,CPC,ROAS,ACOS,Report_Type
0,2026-06-01,B0BG2Q17ZC,NaN,71,0,0.00,0.00,0.00,0,0,0.00,0.00,0.0000,Sponsored Products
1,2026-06-01,B08B4T7CXB,NaN,732,5,0.20,16.42,12.69,1,1,3.28,0.77,1.2939,Sponsored Products
2,2026-06-01,B09KCG9FZT,NaN,312,1,0.00,3.07,0.00,0,0,3.07,0.00,0.0000,Sponsored Products
3,2026-06-01,B09M1MKKJG,NaN,72,3,0.67,10.89,38.07,2,3,3.63,3.50,0.2861,Sponsored Products
4,2026-06-01,B0DDGWMTY5,NaN,155,0,0.00,0.00,0.00,0,0,0.00,0.00,0.0000,Sponsored Products
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2181,2026-06-10,B08B4T7CXB,NaN,6,0,0.00,0.00,0.00,0,0,0.00,0.00,0.0000,Sponsored Products
2182,2026-06-10,B09KCG9FZT,NaN,31,1,0.00,14.39,0.00,0,0,14.39,0.00,0.0000,Sponsored Products
2183,2026-06-10,B09WD3WPF1,NaN,1,0,0.00,0.00,0.00,0,0,0.00,0.00,0.0000,Sponsored Products
2184,2026-06-10,B09WD3WPF1,NaN,5,0,0.00,0.00,0.00,0,0,0.00,0.00,0.0000,Sponsored Products


## convert into float

In [4]:
numeric_cols = [
    "Impressions",
    "Clicks",
    "Conversion Rate",
    "Spend",
    "Sales",
    "Orders",
    "Units",
    "CPC",
    "ROAS",
    "ACOS"
]

for col in numeric_cols:
    if col in df_spap.columns:
        df_spap[col] = (
            df_spap[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("₹", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.strip()
        )

        df_spap[col] = pd.to_numeric(df_spap[col], errors="coerce")

In [5]:
df_sb

,Product,Entity,Operation,Campaign ID,Draft Campaign ID,Portfolio ID,Ad Group ID,Keyword ID,Product Targeting ID,Campaign Name,...,Click-through Rate,Spend,Sales,Orders,Units,Conversion Rate,ACOS,CPC,ROAS,Date
0,Sponsored Brands,Campaign,NaN,416832889387056,NaN,NaN,NaN,NaN,NaN,Pattex | Superglue | SB | PC | KW,...,0.0104,58.96,99.27,5,9,0.21,0.5939,2.46,1.68,2026-06-01
1,Sponsored Brands,Campaign,NaN,375458985898716,NaN,NaN,NaN,NaN,NaN,PATTEX-SB-M(VPP),...,0.0056,102.71,273.16,9,10,0.30,0.3760,3.42,2.66,2026-06-01
2,Sponsored Brands,Campaign,NaN,562187756530334,NaN,NaN,NaN,NaN,NaN,PATTEX | SBV | NMN | CPC | G+BD+BO,...,0.0000,0.00,0.00,0,0,0.00,0.0000,0.00,0.00,2026-06-01
3,Sponsored Brands,Campaign,NaN,482574006456515,NaN,NaN,NaN,NaN,NaN,PATTEX | SBV | Sealant | CPC | G+BD+BO,...,0.0000,0.00,0.00,0,0,0.00,0.0000,0.00,0.00,2026-06-01
4,Sponsored Brands,Campaign,NaN,308236872542027,NaN,NaN,NaN,NaN,NaN,PATTEX | SBV | Superglue | CPC | G+BD+BO,...,0.0000,0.00,0.00,0,0,0.00,0.0000,0.00,0.00,2026-06-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,Sponsored Brands,Campaign,NaN,482574006456515,NaN,NaN,NaN,NaN,NaN,PATTEX | SBV | Sealant | CPC | G+BD+BO,...,0.0065,5.72,0.00,0,0,0.00,0.0000,5.72,0.00,2026-06-10
76,Sponsored Brands,Campaign,NaN,308236872542027,NaN,NaN,NaN,NaN,NaN,PATTEX | SBV | Superglue | CPC | G+BD+BO,...,0.0022,3.60,19.79,2,2,2.00,0.1819,3.60,5.50,2026-06-10
77,Sponsored Brands,Campaign,NaN,358348630592250,NaN,NaN,NaN,NaN,NaN,PATTEX | SBV | Construction Adhesive | CPC | G...,...,0.0000,0.00,0.00,0,0,0.00,0.0000,0.00,0.00,2026-06-10
78,Sponsored Brands,Campaign,NaN,464761005202754,NaN,NaN,NaN,NaN,NaN,PATTEX | SBV | Contact Adhesive | CPC | G+BD+BO,...,0.0241,4.03,35.98,1,2,0.50,0.1120,2.02,8.93,2026-06-10


In [6]:
cols_to_use = [
    "Date",
    "Creative ASINs",
    "Campaign Name",
    "Impressions",
    "Clicks",
    "Conversion Rate",
    "Spend",
    "Sales",
    "Orders",
    "Units",
    "CPC",
    "ROAS",
    "ACOS", 
]
df_sb = df_sb[cols_to_use].copy()
df_sb = df_sb.rename(columns={"Creative ASINs" : "ASIN"})
df_sb["Report_Type"] = "Sponsored Brands"
df_sb

,Date,ASIN,Campaign Name,Impressions,Clicks,Conversion Rate,Spend,Sales,Orders,Units,CPC,ROAS,ACOS,Report_Type
0,2026-06-01,"B09KCFHH3T, B0BG2MZ1ZX, B09KCGPSPG",Pattex | Superglue | SB | PC | KW,2300,24,0.21,58.96,99.27,5,9,2.46,1.68,0.5939,Sponsored Brands
1,2026-06-01,B09KCGPSPG,PATTEX-SB-M(VPP),5377,30,0.30,102.71,273.16,9,10,3.42,2.66,0.3760,Sponsored Brands
2,2026-06-01,B09KCG9FZT,PATTEX | SBV | NMN | CPC | G+BD+BO,67,0,0.00,0.00,0.00,0,0,0.00,0.00,0.0000,Sponsored Brands
3,2026-06-01,B08Y917ZRH,PATTEX | SBV | Sealant | CPC | G+BD+BO,147,0,0.00,0.00,0.00,0,0,0.00,0.00,0.0000,Sponsored Brands
4,2026-06-01,B09KCFHH3T,PATTEX | SBV | Superglue | CPC | G+BD+BO,450,0,0.00,0.00,0.00,0,0,0.00,0.00,0.0000,Sponsored Brands
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,2026-06-10,B08Y917ZRH,PATTEX | SBV | Sealant | CPC | G+BD+BO,154,1,0.00,5.72,0.00,0,0,5.72,0.00,0.0000,Sponsored Brands
76,2026-06-10,B09KCFHH3T,PATTEX | SBV | Superglue | CPC | G+BD+BO,450,1,2.00,3.60,19.79,2,2,3.60,5.50,0.1819,Sponsored Brands
77,2026-06-10,B08MXBVG7Z,PATTEX | SBV | Construction Adhesive | CPC | G...,25,0,0.00,0.00,0.00,0,0,0.00,0.00,0.0000,Sponsored Brands
78,2026-06-10,B08MXD1DCV,PATTEX | SBV | Contact Adhesive | CPC | G+BD+BO,83,2,0.50,4.03,35.98,1,2,2.02,8.93,0.1120,Sponsored Brands


## convert into float

In [7]:
numeric_cols = [
    "Impressions",
    "Clicks",
    "Conversion Rate",
    "Spend",
    "Sales",
    "Orders",
    "Units",
    "CPC",
    "ROAS",
    "ACOS"
]

for col in numeric_cols:
    if col in df_sb.columns:
        df_sb[col] = (
            df_sb[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("₹", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.strip()
        )

        df_sb[col] = pd.to_numeric(df_sb[col], errors="coerce")

## Handle Multiple Asins

In [8]:
numeric_cols = [
    'Impressions',
    'Clicks',
    'Conversion Rate',
    'Spend',
    'Sales',
    'Orders',
    'Units',
    'CPC',
    'ROAS',
    'ACOS'
]

# Create a copy
df_sb_expanded = df_sb.copy()

# Convert ASIN string to list
df_sb_expanded['ASIN'] = (
    df_sb_expanded['ASIN']
    .fillna('')
    .astype(str)
    .str.split(r'\s*,\s*')
)

# Count ASINs per row
df_sb_expanded['asin_count'] = df_sb_expanded['ASIN'].apply(
    lambda x: len([i for i in x if i.strip()])
)

# Divide metrics by number of ASINs
for col in numeric_cols:
    if col in df_sb_expanded.columns:
        df_sb_expanded[col] = (
            df_sb_expanded[col] /
            df_sb_expanded['asin_count']
        )

# Create one row per ASIN
df_sb_expanded = df_sb_expanded.explode('ASIN')

# Remove helper column
df_sb_expanded = df_sb_expanded.drop(columns='asin_count')

# Clean ASIN values
df_sb_expanded['ASIN'] = df_sb_expanded['ASIN'].str.strip()

print(df_sb_expanded.shape)
df_sb_expanded

(100, 14)


,Date,ASIN,Campaign Name,Impressions,Clicks,Conversion Rate,Spend,Sales,Orders,Units,CPC,ROAS,ACOS,Report_Type
0,2026-06-01,B09KCFHH3T,Pattex | Superglue | SB | PC | KW,766.666667,8.0,0.07,19.653333,33.09,1.666667,3.0,0.82,0.56,0.197967,Sponsored Brands
0,2026-06-01,B0BG2MZ1ZX,Pattex | Superglue | SB | PC | KW,766.666667,8.0,0.07,19.653333,33.09,1.666667,3.0,0.82,0.56,0.197967,Sponsored Brands
0,2026-06-01,B09KCGPSPG,Pattex | Superglue | SB | PC | KW,766.666667,8.0,0.07,19.653333,33.09,1.666667,3.0,0.82,0.56,0.197967,Sponsored Brands
1,2026-06-01,B09KCGPSPG,PATTEX-SB-M(VPP),5377.000000,30.0,0.30,102.710000,273.16,9.000000,10.0,3.42,2.66,0.376000,Sponsored Brands
2,2026-06-01,B09KCG9FZT,PATTEX | SBV | NMN | CPC | G+BD+BO,67.000000,0.0,0.00,0.000000,0.00,0.000000,0.0,0.00,0.00,0.000000,Sponsored Brands
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,2026-06-10,B08Y917ZRH,PATTEX | SBV | Sealant | CPC | G+BD+BO,154.000000,1.0,0.00,5.720000,0.00,0.000000,0.0,5.72,0.00,0.000000,Sponsored Brands
76,2026-06-10,B09KCFHH3T,PATTEX | SBV | Superglue | CPC | G+BD+BO,450.000000,1.0,2.00,3.600000,19.79,2.000000,2.0,3.60,5.50,0.181900,Sponsored Brands
77,2026-06-10,B08MXBVG7Z,PATTEX | SBV | Construction Adhesive | CPC | G...,25.000000,0.0,0.00,0.000000,0.00,0.000000,0.0,0.00,0.00,0.000000,Sponsored Brands
78,2026-06-10,B08MXD1DCV,PATTEX | SBV | Contact Adhesive | CPC | G+BD+BO,83.000000,2.0,0.50,4.030000,35.98,1.000000,2.0,2.02,8.93,0.112000,Sponsored Brands


In [9]:
df_final = pd.concat(
    [df_spap, df_sb_expanded],
    ignore_index=True
)
numeric_cols = df_final.select_dtypes(include='number').columns

df_final[numeric_cols] = df_final[numeric_cols].round(2)
df_final

,Date,ASIN,Campaign Name,Impressions,Clicks,Conversion Rate,Spend,Sales,Orders,Units,CPC,ROAS,ACOS,Report_Type
0,2026-06-01,B0BG2Q17ZC,NaN,71.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,Sponsored Products
1,2026-06-01,B08B4T7CXB,NaN,732.0,5.0,0.20,16.42,12.69,1.0,1.0,3.28,0.77,1.29,Sponsored Products
2,2026-06-01,B09KCG9FZT,NaN,312.0,1.0,0.00,3.07,0.00,0.0,0.0,3.07,0.00,0.00,Sponsored Products
3,2026-06-01,B09M1MKKJG,NaN,72.0,3.0,0.67,10.89,38.07,2.0,3.0,3.63,3.50,0.29,Sponsored Products
4,2026-06-01,B0DDGWMTY5,NaN,155.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,Sponsored Products
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2281,2026-06-10,B08Y917ZRH,PATTEX | SBV | Sealant | CPC | G+BD+BO,154.0,1.0,0.00,5.72,0.00,0.0,0.0,5.72,0.00,0.00,Sponsored Brands
2282,2026-06-10,B09KCFHH3T,PATTEX | SBV | Superglue | CPC | G+BD+BO,450.0,1.0,2.00,3.60,19.79,2.0,2.0,3.60,5.50,0.18,Sponsored Brands
2283,2026-06-10,B08MXBVG7Z,PATTEX | SBV | Construction Adhesive | CPC | G...,25.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,Sponsored Brands
2284,2026-06-10,B08MXD1DCV,PATTEX | SBV | Contact Adhesive | CPC | G+BD+BO,83.0,2.0,0.50,4.03,35.98,1.0,2.0,2.02,8.93,0.11,Sponsored Brands


In [10]:
df_final

,Date,ASIN,Campaign Name,Impressions,Clicks,Conversion Rate,Spend,Sales,Orders,Units,CPC,ROAS,ACOS,Report_Type
0,2026-06-01,B0BG2Q17ZC,NaN,71.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,Sponsored Products
1,2026-06-01,B08B4T7CXB,NaN,732.0,5.0,0.20,16.42,12.69,1.0,1.0,3.28,0.77,1.29,Sponsored Products
2,2026-06-01,B09KCG9FZT,NaN,312.0,1.0,0.00,3.07,0.00,0.0,0.0,3.07,0.00,0.00,Sponsored Products
3,2026-06-01,B09M1MKKJG,NaN,72.0,3.0,0.67,10.89,38.07,2.0,3.0,3.63,3.50,0.29,Sponsored Products
4,2026-06-01,B0DDGWMTY5,NaN,155.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,Sponsored Products
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2281,2026-06-10,B08Y917ZRH,PATTEX | SBV | Sealant | CPC | G+BD+BO,154.0,1.0,0.00,5.72,0.00,0.0,0.0,5.72,0.00,0.00,Sponsored Brands
2282,2026-06-10,B09KCFHH3T,PATTEX | SBV | Superglue | CPC | G+BD+BO,450.0,1.0,2.00,3.60,19.79,2.0,2.0,3.60,5.50,0.18,Sponsored Brands
2283,2026-06-10,B08MXBVG7Z,PATTEX | SBV | Construction Adhesive | CPC | G...,25.0,0.0,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,Sponsored Brands
2284,2026-06-10,B08MXD1DCV,PATTEX | SBV | Contact Adhesive | CPC | G+BD+BO,83.0,2.0,0.50,4.03,35.98,1.0,2.0,2.02,8.93,0.11,Sponsored Brands


In [11]:
df_final.to_csv(
    r"C:\Users\Shubham Verma\OneDrive - Incircle Ecoproducts Pvt Ltd\Thrivebrands - Client Database\Pattex\Ads\pattex\vendor_central\output\pattex_vendor_central_ads.csv",
    index=False
)